In [16]:
# ============================================================
# SETUP
# ============================================================

from pathlib import Path
import json
import re
import numpy as np
import pandas as pd

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import wilcoxon

# Change this if needed
PROJECT_DIR = Path(r"C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility")
DATA_DIR = PROJECT_DIR / "processed_data_final"

FPS = 25
HISTORY = 25
STRIDE = 1

HORIZONS = [1, 15, 25, 50, 75, 125]

BRAKE_THRESHOLD = -0.1
ACCEL_THRESHOLD = 0.2

THRESHOLD_TAG = "brake0p1_accel0p2"
IDM_PARAM_TAG = "refinedIDM_v2"

PERSISTENCE_ROOT = DATA_DIR / "persistence_baseline_runs"
PERSISTENCE_ROOT.mkdir(parents=True, exist_ok=True)

SIGNIFICANCE_OUT_DIR = DATA_DIR / "significance_tests_by_horizon"
SIGNIFICANCE_OUT_DIR.mkdir(parents=True, exist_ok=True)


def horizon_tag(h):
    return f"H{HISTORY}_Hz{h}_S{STRIDE}"


def horizon_seconds(h):
    return h / FPS


def horizon_label(h):
    return f"{h / FPS:.2f}s"


print("DATA_DIR:", DATA_DIR)
print("Horizons:", HORIZONS)

DATA_DIR: C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final
Horizons: [1, 15, 25, 50, 75, 125]


In [17]:
# ============================================================
# LOAD RAW TEST DATA
# ============================================================

def read_any_table(path):
    path = Path(path)

    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path)

    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)

    if path.suffix.lower() in [".pkl", ".pickle"]:
        return pd.read_pickle(path)

    raise ValueError(f"Unsupported file type: {path}")


def find_raw_test_file():
    candidates = []

    patterns = [
        "*test*.parquet",
        "*test*.csv",
        "*test*.pkl",
        "*test*.pickle",
    ]

    for pattern in patterns:
        candidates.extend(DATA_DIR.rglob(pattern))

    # Prefer files that look like the processed test split, not predictions/metrics/logs
    bad_words = [
        "prediction",
        "predictions",
        "metrics",
        "summary",
        "training",
        "log",
        "result",
        "results",
        "safety",
        "regime",
    ]

    candidates = [
        p for p in candidates
        if not any(bad in p.name.lower() for bad in bad_words)
    ]

    checked = []

    for p in candidates:
        try:
            df_head = read_any_table(p).head(5)
            cols = set(df_head.columns)

            has_episode = "episode_id" in cols or "unique_id" in cols
            has_frame = "frame" in cols
            has_accel = (
                "a_true" in cols
                or "xAcceleration" in cols
                or "acceleration" in cols
            )

            if has_episode and has_frame and has_accel:
                checked.append(p)

        except Exception:
            pass

    if len(checked) == 0:
        raise FileNotFoundError(
            "Could not automatically find the raw test file. "
            "Set TEST_RAW_PATH manually."
        )

    # Pick newest valid file
    return max(checked, key=lambda p: p.stat().st_mtime)


# Manual override if needed:
# TEST_RAW_PATH = DATA_DIR / "your_test_file.parquet"

try:
    TEST_RAW_PATH
except NameError:
    TEST_RAW_PATH = find_raw_test_file()

test_raw = read_any_table(TEST_RAW_PATH)

print("Loaded raw test file:")
print(TEST_RAW_PATH)
print(test_raw.shape)
print(test_raw.columns.tolist())

# Standardize acceleration column
if "a_true" not in test_raw.columns:
    if "xAcceleration" in test_raw.columns:
        test_raw["a_true"] = test_raw["xAcceleration"]
    elif "acceleration" in test_raw.columns:
        test_raw["a_true"] = test_raw["acceleration"]
    else:
        raise ValueError("No acceleration column found in raw test data.")

# Standardize episode key
if "episode_id" not in test_raw.columns:
    if "unique_id" in test_raw.columns:
        test_raw["episode_id"] = test_raw["unique_id"]
    else:
        raise ValueError("No episode_id or unique_id column found.")

test_raw["frame"] = test_raw["frame"].astype(int)
test_raw["a_true"] = pd.to_numeric(test_raw["a_true"], errors="coerce").astype(np.float32)

# Optional context columns
for col in ["gap", "v_ego", "v_leader", "delta_v"]:
    if col in test_raw.columns:
        test_raw[col] = pd.to_numeric(test_raw[col], errors="coerce").astype(np.float32)

test_raw_lookup = test_raw[
    [
        c for c in [
            "episode_id",
            "frame",
            "a_true",
            "gap",
            "v_ego",
            "v_leader",
            "delta_v",
        ]
        if c in test_raw.columns
    ]
].copy()

print("Prepared raw lookup:", test_raw_lookup.shape)

Loaded raw test file:
C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\hybrid_idm_first_runs\Hybrid_OracleRegimeAwareIDM_H25_Hz75_S1_brake0p1_accel0p2_refinedIDM_v2\test_sequences.meta.csv
(1628545, 16)
['episode_id', 'unique_id', 'input_frame', 'target_frame', 'frame', 'a_true', 'a_idm_prior', 'history_frames', 'horizon_frames', 'horizon_seconds', 'stride', 'a_idm_global', 'a_idm_braking', 'a_idm_accel', 'a_idm_steady', 'regime']
Prepared raw lookup: (1628545, 3)


In [18]:
# ============================================================
# ACCELERATION PERSISTENCE BASELINE
#
# y_pred(t + horizon) = acceleration(t)
#
# This version builds the persistence baseline directly from test_raw.
# ============================================================

from sklearn.metrics import r2_score

def compute_basic_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)

    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    err = y_pred - y_true
    abs_err = np.abs(err)

    if len(y_true) == 0:
        return {}

    corr = np.corrcoef(y_true, y_pred)[0, 1] if len(y_true) > 1 else np.nan

    return {
        "Count": int(len(y_true)),
        "MAE": float(np.mean(abs_err)),
        "RMSE": float(np.sqrt(np.mean(err ** 2))),
        "MedAE": float(np.median(abs_err)),
        "P95_abs_error": float(np.percentile(abs_err, 95)),
        "P99_abs_error": float(np.percentile(abs_err, 99)),
        "R2": float(r2_score(y_true, y_pred)),
        "Bias": float(np.mean(err)),
        "Corr": float(corr),
    }


def add_regime(df):
    out = df.copy()

    out["regime"] = np.where(
        out["y_true"] < BRAKE_THRESHOLD,
        "braking",
        np.where(out["y_true"] > ACCEL_THRESHOLD, "accelerating", "steady")
    )

    return out


def compute_regime_metrics(df):
    rows = []

    for regime, g in df.groupby("regime"):
        metrics = compute_basic_metrics(g["y_true"], g["y_pred"])
        metrics["Regime"] = regime
        rows.append(metrics)

    return pd.DataFrame(rows)


def compute_brake_safety_metrics(df):
    y_true = df["y_true"].to_numpy()
    y_pred = df["y_pred"].to_numpy()

    err = y_pred - y_true
    abs_err = np.abs(err)

    braking_mask = y_true < BRAKE_THRESHOLD

    out = {
        "MAE": float(np.mean(abs_err)),
        "RMSE": float(np.sqrt(np.mean(err ** 2))),
        "P95_abs_error": float(np.percentile(abs_err, 95)),
        "P99_abs_error": float(np.percentile(abs_err, 99)),
    }

    if braking_mask.sum() > 0:
        brake_true = y_true[braking_mask]
        brake_pred = y_pred[braking_mask]
        brake_err = brake_pred - brake_true

        out["Braking_MAE"] = float(np.mean(np.abs(brake_err)))

        # True braking is negative.
        # Under-braking means the prediction is not negative enough.
        out["UnderBrake_Rate_gt_0p5"] = float(np.mean(brake_err > 0.5))

        # Wrong sign during true braking.
        out["WrongSign_Rate_in_Braking"] = float(np.mean(brake_pred >= 0.0))

        # Brake recall.
        out["BrakeRecall"] = float(np.mean(brake_pred < BRAKE_THRESHOLD))
    else:
        out["Braking_MAE"] = np.nan
        out["UnderBrake_Rate_gt_0p5"] = np.nan
        out["WrongSign_Rate_in_Braking"] = np.nan
        out["BrakeRecall"] = np.nan

    out["SignAccuracy"] = float(np.mean(np.sign(y_true) == np.sign(y_pred)))

    return out


def build_persistence_sequences_from_raw(df, h):
    """
    Builds the same horizon logic as the sequence models:

    history window ends at current/input frame t
    target frame = t + horizon
    y_pred = acceleration at t
    y_true = acceleration at t + horizon
    """

    rows = []

    needed_cols = ["episode_id", "frame", "a_true"]
    for c in needed_cols:
        if c not in df.columns:
            raise ValueError(f"Missing required column in test_raw: {c}")

    context_cols = [
        c for c in ["gap", "v_ego", "v_leader", "delta_v"]
        if c in df.columns
    ]

    for episode_id, g in df.groupby("episode_id", sort=False):
        g = g.sort_values("frame").reset_index(drop=True)

        # Require continuous frames, same logic as the model sequence builders.
        frame_diff = g["frame"].diff().dropna()
        if len(frame_diff) > 0 and not (frame_diff == 1).all():
            continue

        n = len(g)

        if n < HISTORY + h:
            continue

        current_idx = np.arange(HISTORY - 1, n - h, STRIDE, dtype=int)
        target_idx = current_idx + h

        current = g.iloc[current_idx].copy().reset_index(drop=True)
        target = g.iloc[target_idx].copy().reset_index(drop=True)

        part = pd.DataFrame({
            "Model": "Acceleration Persistence",
            "HorizonFrames": h,
            "HorizonSeconds": h / FPS,
            "HorizonLabel": horizon_label(h),
            "episode_id": episode_id,
            "input_frame": current["frame"].to_numpy(dtype=int),
            "target_frame": target["frame"].to_numpy(dtype=int),
            "frame": target["frame"].to_numpy(dtype=int),
            "a_current": current["a_true"].to_numpy(dtype=np.float32),
            "y_true": target["a_true"].to_numpy(dtype=np.float32),
            "y_pred": current["a_true"].to_numpy(dtype=np.float32),
        })

        for col in context_cols:
            part[f"{col}_input"] = current[col].to_numpy(dtype=np.float32)

        rows.append(part)

    if len(rows) == 0:
        raise ValueError(f"No persistence rows created for horizon {h}.")

    out = pd.concat(rows, ignore_index=True)

    # Strong check: target frame must be exactly horizon frames after input frame.
    frame_shift = out["target_frame"] - out["input_frame"]
    assert (frame_shift == h).all(), f"Frame shift mismatch for horizon {h}"

    out = add_regime(out)

    return out


def build_persistence_for_horizon(h):
    tag = horizon_tag(h)

    pred = build_persistence_sequences_from_raw(test_raw, h)

    run_dir = PERSISTENCE_ROOT / f"AccelerationPersistence_{tag}_{THRESHOLD_TAG}"
    run_dir.mkdir(parents=True, exist_ok=True)

    pred_path = run_dir / f"persistence_test_predictions_{tag}.csv"
    summary_path = run_dir / f"persistence_summary_{tag}.csv"
    regime_path = run_dir / f"persistence_regime_metrics_{tag}.csv"
    safety_path = run_dir / f"persistence_brake_safety_{tag}.csv"

    pred.to_csv(pred_path, index=False)

    global_metrics = compute_basic_metrics(pred["y_true"], pred["y_pred"])
    global_metrics.update({
        "Model": "Acceleration Persistence",
        "ShortName": "Persistence",
        "Family": "Baseline",
        "Type": "Deployable",
        "HorizonFrames": h,
        "HorizonSeconds": h / FPS,
        "HorizonLabel": horizon_label(h),
        "PredictionFile": str(pred_path),
    })

    pd.DataFrame([global_metrics]).to_csv(summary_path, index=False)

    regime_metrics = compute_regime_metrics(pred)
    regime_metrics["Model"] = "Acceleration Persistence"
    regime_metrics["ShortName"] = "Persistence"
    regime_metrics["Family"] = "Baseline"
    regime_metrics["Type"] = "Deployable"
    regime_metrics["HorizonFrames"] = h
    regime_metrics["HorizonSeconds"] = h / FPS
    regime_metrics["HorizonLabel"] = horizon_label(h)
    regime_metrics.to_csv(regime_path, index=False)

    safety_metrics = compute_brake_safety_metrics(pred)
    safety_metrics.update({
        "Model": "Acceleration Persistence",
        "ShortName": "Persistence",
        "Family": "Baseline",
        "Type": "Deployable",
        "HorizonFrames": h,
        "HorizonSeconds": h / FPS,
        "HorizonLabel": horizon_label(h),
        "PredictionFile": str(pred_path),
    })

    pd.DataFrame([safety_metrics]).to_csv(safety_path, index=False)

    return {
        "HorizonFrames": h,
        "HorizonSeconds": h / FPS,
        "HorizonLabel": horizon_label(h),
        "Rows": len(pred),
        "PredictionFile": str(pred_path),
        "SummaryFile": str(summary_path),
        "RegimeFile": str(regime_path),
        "SafetyFile": str(safety_path),
        "MAE": global_metrics["MAE"],
        "RMSE": global_metrics["RMSE"],
        "MedAE": global_metrics["MedAE"],
        "P95_abs_error": global_metrics["P95_abs_error"],
        "P99_abs_error": global_metrics["P99_abs_error"],
        "R2": global_metrics["R2"],
        "Bias": global_metrics["Bias"],
        "Corr": global_metrics["Corr"],
    }


persistence_rows = []

for h in HORIZONS:
    print("\n" + "=" * 100)
    print("Building persistence baseline for:", horizon_tag(h))
    print("=" * 100)

    row = build_persistence_for_horizon(h)
    persistence_rows.append(row)

persistence_manifest = pd.DataFrame(persistence_rows)

manifest_path = PERSISTENCE_ROOT / "persistence_baseline_manifest.csv"
persistence_manifest.to_csv(manifest_path, index=False)

display(persistence_manifest)

print("Saved persistence manifest:")
print(manifest_path)


Building persistence baseline for: H25_Hz1_S1

Building persistence baseline for: H25_Hz15_S1

Building persistence baseline for: H25_Hz25_S1

Building persistence baseline for: H25_Hz50_S1

Building persistence baseline for: H25_Hz75_S1

Building persistence baseline for: H25_Hz125_S1


,HorizonFrames,HorizonSeconds,HorizonLabel,Rows,PredictionFile,SummaryFile,RegimeFile,SafetyFile,MAE,RMSE,MedAE,P95_abs_error,P99_abs_error,R2,Bias,Corr
0,1,0.04,0.04s,1441770,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...,0.004040,0.007448,0.00,0.01,0.02,0.999517,0.000028,0.999758
1,15,0.60,0.60s,1337920,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...,0.060298,0.095567,0.04,0.20,0.35,0.920836,0.000432,0.960520
2,25,1.00,1.00s,1264871,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...,0.098471,0.155946,0.06,0.32,0.56,0.789877,0.000801,0.895541
3,50,2.00,2.00s,1086466,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...,0.179424,0.283809,0.11,0.59,1.02,0.312592,0.002541,0.662036
4,75,3.00,3.00s,915477,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...,0.236635,0.372465,0.14,0.77,1.35,-0.156107,0.005389,0.438774
5,125,5.00,5.00s,607156,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...,0.305508,0.469627,0.19,0.97,1.68,-0.718185,0.009824,0.194343


Saved persistence manifest:
C:\Users\ANARAMO\Documents\Tese\Tese\Tese\HybridMobility\processed_data_final\persistence_baseline_runs\persistence_baseline_manifest.csv


In [19]:
# ============================================================
# ADD PERSISTENCE BASELINE TO FINAL TABLE INPUTS
# ============================================================

def load_persistence_outputs():
    global_rows = []
    regime_rows = []
    safety_rows = []

    for h in HORIZONS:
        tag = horizon_tag(h)
        run_dir = PERSISTENCE_ROOT / f"AccelerationPersistence_{tag}_{THRESHOLD_TAG}"

        summary_file = run_dir / f"persistence_summary_{tag}.csv"
        regime_file = run_dir / f"persistence_regime_metrics_{tag}.csv"
        safety_file = run_dir / f"persistence_brake_safety_{tag}.csv"

        if summary_file.exists():
            global_rows.append(pd.read_csv(summary_file))

        if regime_file.exists():
            regime_rows.append(pd.read_csv(regime_file))

        if safety_file.exists():
            safety_rows.append(pd.read_csv(safety_file))

    persistence_global = pd.concat(global_rows, ignore_index=True) if global_rows else pd.DataFrame()
    persistence_regime = pd.concat(regime_rows, ignore_index=True) if regime_rows else pd.DataFrame()
    persistence_safety = pd.concat(safety_rows, ignore_index=True) if safety_rows else pd.DataFrame()

    return persistence_global, persistence_regime, persistence_safety


persistence_global, persistence_regime, persistence_safety = load_persistence_outputs()

print("Persistence global:", persistence_global.shape)
print("Persistence regime:", persistence_regime.shape)
print("Persistence safety:", persistence_safety.shape)

display(persistence_global)

Persistence global: (6, 17)
Persistence regime: (18, 17)
Persistence safety: (6, 17)


,Count,MAE,RMSE,MedAE,P95_abs_error,P99_abs_error,R2,Bias,Corr,Model,ShortName,Family,Type,HorizonFrames,HorizonSeconds,HorizonLabel,PredictionFile
0,1441770,0.004040,0.007448,0.00,0.01,0.02,0.999517,0.000028,0.999758,Acceleration Persistence,Persistence,Baseline,Deployable,1,0.04,0.04s,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...
1,1337920,0.060298,0.095567,0.04,0.20,0.35,0.920836,0.000432,0.960520,Acceleration Persistence,Persistence,Baseline,Deployable,15,0.60,0.60s,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...
2,1264871,0.098471,0.155946,0.06,0.32,0.56,0.789877,0.000801,0.895541,Acceleration Persistence,Persistence,Baseline,Deployable,25,1.00,1.00s,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...
3,1086466,0.179424,0.283809,0.11,0.59,1.02,0.312592,0.002541,0.662036,Acceleration Persistence,Persistence,Baseline,Deployable,50,2.00,2.00s,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...
4,915477,0.236635,0.372465,0.14,0.77,1.35,-0.156107,0.005389,0.438774,Acceleration Persistence,Persistence,Baseline,Deployable,75,3.00,3.00s,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...
5,607156,0.305508,0.469627,0.19,0.97,1.68,-0.718185,0.009824,0.194343,Acceleration Persistence,Persistence,Baseline,Deployable,125,5.00,5.00s,C:\Users\ANARAMO\Documents\Tese\Tese\Tese\Hybr...


In [20]:
# ============================================================
# EPISODE-LEVEL SIGNIFICANCE TESTS
# ============================================================

def standardize_prediction_file_for_tests(path, model_name, h):
    df = pd.read_csv(path)

    out = df.copy()

    if "episode_id" not in out.columns:
        if "unique_id" in out.columns:
            out["episode_id"] = out["unique_id"]
        else:
            raise ValueError(f"{model_name} missing episode_id/unique_id.")

    if "y_true" not in out.columns:
        for c in ["a_true", "target", "true_accel"]:
            if c in out.columns:
                out["y_true"] = out[c]
                break

    if "y_pred" not in out.columns:
        for c in [
            "prediction",
            "a_pred",
            "pred",
            "test_pred",
            "y_hat",
            "a_idm",
            "a_idm_piecewise",
            "a_idm_best_candidate",
            "a_idm_soft",
            "a_hybrid",
            "weighted_prediction",
            "weighted_pred",
            "a_fused",
        ]:
            if c in out.columns:
                out["y_pred"] = out[c]
                break

    if "y_true" not in out.columns or "y_pred" not in out.columns:
        raise ValueError(
            f"{model_name} missing y_true/y_pred. Columns: {list(out.columns)}"
        )

    out["y_true"] = pd.to_numeric(out["y_true"], errors="coerce")
    out["y_pred"] = pd.to_numeric(out["y_pred"], errors="coerce")

    out = out.dropna(subset=["y_true", "y_pred"]).copy()

    out["abs_error"] = np.abs(out["y_pred"] - out["y_true"])
    out["squared_error"] = (out["y_pred"] - out["y_true"]) ** 2
    out["Model"] = model_name
    out["HorizonFrames"] = h
    out["HorizonSeconds"] = h / FPS
    out["HorizonLabel"] = horizon_label(h)

    return out


def episode_level_errors(df):
    return (
        df.groupby("episode_id", sort=False)
        .agg(
            EpisodeMAE=("abs_error", "mean"),
            EpisodeRMSE=("squared_error", lambda x: np.sqrt(np.mean(x))),
            Count=("abs_error", "size"),
        )
        .reset_index()
    )


def paired_bootstrap_ci(diff, n_boot=5000, seed=42, ci=95):
    rng = np.random.default_rng(seed)
    diff = np.asarray(diff, dtype=np.float64)
    diff = diff[np.isfinite(diff)]

    if len(diff) == 0:
        return np.nan, np.nan

    boot_means = np.empty(n_boot)

    n = len(diff)

    for i in range(n_boot):
        sample = rng.choice(diff, size=n, replace=True)
        boot_means[i] = np.mean(sample)

    alpha = (100 - ci) / 2

    return (
        float(np.percentile(boot_means, alpha)),
        float(np.percentile(boot_means, 100 - alpha)),
    )


def paired_episode_test(df_a, df_b, model_a, model_b, h):
    ep_a = episode_level_errors(df_a)
    ep_b = episode_level_errors(df_b)

    merged = ep_a.merge(
        ep_b,
        on="episode_id",
        suffixes=("_A", "_B"),
        how="inner",
    )

    if len(merged) < 5:
        return {
            "HorizonFrames": h,
            "HorizonSeconds": h / FPS,
            "HorizonLabel": horizon_label(h),
            "ModelA": model_a,
            "ModelB": model_b,
            "N_Episodes": len(merged),
            "Error": "Too few matched episodes",
        }

    # Difference is A - B.
    # Negative means Model A has lower MAE than Model B.
    diff = merged["EpisodeMAE_A"] - merged["EpisodeMAE_B"]

    mean_diff = float(diff.mean())
    median_diff = float(diff.median())

    try:
        stat, p_value = wilcoxon(diff)
    except Exception:
        stat, p_value = np.nan, np.nan

    ci_low, ci_high = paired_bootstrap_ci(diff)

    return {
        "HorizonFrames": h,
        "HorizonSeconds": h / FPS,
        "HorizonLabel": horizon_label(h),
        "ModelA": model_a,
        "ModelB": model_b,
        "N_Episodes": int(len(merged)),
        "ModelA_episode_MAE_mean": float(merged["EpisodeMAE_A"].mean()),
        "ModelB_episode_MAE_mean": float(merged["EpisodeMAE_B"].mean()),
        "MeanDiff_A_minus_B": mean_diff,
        "MedianDiff_A_minus_B": median_diff,
        "BootstrapCI95_Low": ci_low,
        "BootstrapCI95_High": ci_high,
        "WilcoxonStatistic": float(stat) if pd.notna(stat) else np.nan,
        "WilcoxonPValue": float(p_value) if pd.notna(p_value) else np.nan,
        "Interpretation": (
            f"{model_a} lower MAE"
            if mean_diff < 0
            else f"{model_b} lower MAE"
        ),
    }

In [21]:
# ============================================================
# DISCOVER PREDICTION FILES FOR SIGNIFICANCE TESTS
# Safer version: does not crash if one model/horizon is missing
# ============================================================

def newest(paths):
    paths = [Path(p) for p in paths if p is not None and Path(p).exists()]
    if not paths:
        return None
    return max(paths, key=lambda p: p.stat().st_mtime)


def csv_has_prediction_columns(path):
    try:
        cols = pd.read_csv(path, nrows=0).columns.tolist()
    except Exception:
        return False

    cols_lower = [c.lower() for c in cols]

    has_true = any(
        c.lower() in cols_lower
        for c in ["y_true", "a_true", "target", "true_accel"]
    )

    has_pred = any(
        c.lower() in cols_lower
        for c in [
            "y_pred",
            "prediction",
            "a_pred",
            "pred",
            "test_pred",
            "y_hat",
            "a_fused",
            "weighted_prediction",
            "weighted_pred",
            "a_hybrid",
            "a_idm",
            "a_idm_prior",
            "a_idm_soft",
            "a_idm_best_candidate",
        ]
    )

    has_episode = any(
        c.lower() in cols_lower
        for c in ["episode_id", "unique_id"]
    )

    return has_true and has_pred and has_episode


def broad_find_prediction_file(h, include_words=None, exclude_words=None):
    tag = horizon_tag(h)

    include_words = include_words or []
    exclude_words = exclude_words or []

    candidates = list(DATA_DIR.rglob(f"*{tag}*.csv"))

    filtered =}*.csv"))

    filtered []

    for p in candidates:
        low = str(p).lower()

        if include_words:
            if not all(w.lower() in low for w in include_words):
                continue

        if exclude_words:
            if any(w.lower() in low for w in exclude_words):
                continue

        if any(
            bad in p.name.lower()
            for bad in ["summary", "metrics", "training_log", "manifest"]
        ):
            continue

        if csv_has_prediction_columns(p):
            filtered.append(p)

    return newest(filtered)


def find_plain_transformer_prediction_file_safe(h):
    tag = horizon_tag(h)

    # First try likely baseline folders/files
    candidates = []

    likely_roots = [
        DATA_DIR / "transformer_baseline_runs",
        DATA_DIR / "transformer_runs",
        DATA_DIR,
    ]

    patterns = [
        f"*plain*transformer*{tag}*.csv",
        f"*transformer*baseline*{tag}*.csv",
        f"*baseline*transformer*{tag}*.csv",
        f"*transformer*test*predictions*{tag}*.csv",
        f"*test_predictions*{tag}*.csv",
    ]

    bad_words = [
        "idm_feature",
        "weighted_fusion",
        "fusion",
        "hybrid",
        "idm_first",
        "global_idm",
        "tree",
        "oracle",
        "true_regime",
        "selector",
        "xgb",
        "regime",
        "summary",
        "metrics",
        "training_log",
        "manifest",
    ]

    for root in likely_roots:
        if not root.exists():
            continue

        for pattern in patterns:
            candidates.extend(root.rglob(pattern))

    valid = []

    for p in candidates:
        low = str(p).lower()

        if any(bad in low for bad in bad_words):
            continue

        if csv_has_prediction_columns(p):
            valid.append(p)

    if valid:
        return newest(valid)

    # Last broad attempt
    return broad_find_prediction_file(
        h,
        include_words=["transformer"],
        exclude_words=bad_words,
    )


def find_prediction_by_model_and_horizon(model_name, h):
    tag = horizon_tag(h)

    # -------------------------
    # Persistence
    # -------------------------
    if model_name == "Acceleration Persistence":
        p = (
            PERSISTENCE_ROOT
            / f"AccelerationPersistence_{tag}_{THRESHOLD_TAG}"
            / f"persistence_test_predictions_{tag}.csv"
        )
        return p if p.exists() else None

    # -------------------------
    # Plain Transformer
    # -------------------------
    if model_name == "Plain Transformer":
        return find_plain_transformer_prediction_file_safe(h)

    # -------------------------
    # Transformer + XGB-soft best-IDM feature
    # -------------------------
    if model_name == "Transformer + XGB-soft best-IDM feature":
        return broad_find_prediction_file(
            h,
            include_words=["transformer_idm_feature", "xgb"],
            exclude_words=["summary", "metrics", "training_log"],
        )

    # -------------------------
    # Weighted Fusion + Global IDM
    # -------------------------
    if model_name == "Transformer Weighted Fusion + Global IDM":
        return broad_find_prediction_file(
            h,
            include_words=["weighted_fusion", "globalidm"],
            exclude_words=["summary", "metrics", "training_log", "regime"],
        )

    # -------------------------
    # Residual Hybrid + Global IDM
    # -------------------------
    if model_name == "Residual Hybrid + Global IDM":
        return broad_find_prediction_file(
            h,
            include_words=["hybrid", "globalidm"],
            exclude_words=["summary", "metrics", "training_log", "regime"],
        )

    # -------------------------
    # Residual Hybrid + XGB-soft best-IDM
    # -------------------------
    if model_name == "Residual Hybrid + XGB-soft best-IDM":
        return broad_find_prediction_file(
            h,
            include_words=["hybrid", "xgbsoftbestidm"],
            exclude_words=["summary", "metrics", "training_log", "regime"],
        )

    # -------------------------
    # Global IDM
    # -------------------------
    if model_name == "Global IDM":
        return broad_find_prediction_file(
            h,
            include_words=["global_idm"],
            exclude_words=["summary", "metrics", "training_log", "regime"],
        )

    return None


SIGNIFICANCE_MODELS = [
    "Acceleration Persistence",
    "Plain Transformer",
    "Transformer + XGB-soft best-IDM feature",
    "Transformer Weighted Fusion + Global IDM",
    "Residual Hybrid + Global IDM",
    "Residual Hybrid + XGB-soft best-IDM",
    "Global IDM",
]

prediction_file_rows = []

for h in HORIZONS:
    for model_name in SIGNIFICANCE_MODELS:
        path = find_prediction_by_model_and_horizon(model_name, h)

        prediction_file_rows.append({
            "HorizonFrames": h,
            "HorizonSeconds": h / FPS,
            "HorizonLabel": horizon_label(h),
            "Model": model_name,
            "PredictionFile": str(path) if path is not None else np.nan,
            "Found": path is not None and Path(path).exists(),
        })

sig_file_manifest = pd.DataFrame(prediction_file_rows)

sig_file_manifest_path = SIGNIFICANCE_OUT_DIR / "significance_prediction_file_manifest.csv"
sig_file_manifest.to_csv(sig_file_manifest_path, index=False)

display(sig_file_manifest)

print("Saved:")
print(sig_file_manifest_path)

print("\nMissing files:")
display(sig_file_manifest[sig_file_manifest["Found"] == False])

SyntaxError: unmatched '}' (644224446.py, line 62)

In [ ]:
# ============================================================
# RUN EPISODE-LEVEL SIGNIFICANCE TESTS
# ============================================================

# Main comparisons.
# ModelA - ModelB is reported.
# Negative MeanDiff means ModelA has lower episode-level MAE.
COMPARISONS = [
    ("Plain Transformer", "Acceleration Persistence"),
    ("Plain Transformer", "Global IDM"),
    ("Plain Transformer", "Transformer + XGB-soft best-IDM feature"),
    ("Plain Transformer", "Transformer Weighted Fusion + Global IDM"),
    ("Plain Transformer", "Residual Hybrid + Global IDM"),
    ("Transformer + XGB-soft best-IDM feature", "Acceleration Persistence"),
    ("Transformer Weighted Fusion + Global IDM", "Acceleration Persistence"),
    ("Residual Hybrid + Global IDM", "Acceleration Persistence"),
]

all_test_rows = []

for h in HORIZONS:
    print("\n" + "=" * 100)
    print("Horizon:", horizon_tag(h))
    print("=" * 100)

    loaded = {}

    for model_name in SIGNIFICANCE_MODELS:
        row = sig_file_manifest[
            (sig_file_manifest["HorizonFrames"] == h)
            & (sig_file_manifest["Model"] == model_name)
        ]

        if len(row) == 0 or not bool(row.iloc[0]["Found"]):
            continue

        path = Path(row.iloc[0]["PredictionFile"])

        try:
            loaded[model_name] = standardize_prediction_file_for_tests(
                path,
                model_name,
                h,
            )
            print("Loaded:", model_name, loaded[model_name].shape)

        except Exception as e:
            print("Could not load:", model_name)
            print(e)

    for model_a, model_b in COMPARISONS:
        if model_a not in loaded or model_b not in loaded:
            print("Skipping comparison:", model_a, "vs", model_b)
            continue

        result = paired_episode_test(
            loaded[model_a],
            loaded[model_b],
            model_a,
            model_b,
            h,
        )

        all_test_rows.append(result)

significance_results = pd.DataFrame(all_test_rows)

significance_path = SIGNIFICANCE_OUT_DIR / "episode_level_significance_tests.csv"
significance_results.to_csv(significance_path, index=False)

print("\nSaved significance results:")
print(significance_path)

display(significance_results)

In [ ]:
# ============================================================
# THESIS-READY SIGNIFICANCE TABLE
# ============================================================

sig_table = significance_results.copy()

# Smaller p-values display nicely
sig_table["WilcoxonPValueFormatted"] = sig_table["WilcoxonPValue"].apply(
    lambda x: "<1e-6" if pd.notna(x) and x < 1e-6 else f"{x:.3g}" if pd.notna(x) else np.nan
)

sig_table["Significant_0p05"] = sig_table["WilcoxonPValue"] < 0.05

sig_table = sig_table[
    [
        "HorizonFrames",
        "HorizonSeconds",
        "HorizonLabel",
        "ModelA",
        "ModelB",
        "N_Episodes",
        "ModelA_episode_MAE_mean",
        "ModelB_episode_MAE_mean",
        "MeanDiff_A_minus_B",
        "BootstrapCI95_Low",
        "BootstrapCI95_High",
        "WilcoxonPValueFormatted",
        "Significant_0p05",
        "Interpretation",
    ]
].copy()

numeric_cols = [
    "HorizonSeconds",
    "ModelA_episode_MAE_mean",
    "ModelB_episode_MAE_mean",
    "MeanDiff_A_minus_B",
    "BootstrapCI95_Low",
    "BootstrapCI95_High",
]

sig_table[numeric_cols] = sig_table[numeric_cols].round(5)

sig_table_path = SIGNIFICANCE_OUT_DIR / "thesis_ready_significance_table.csv"
sig_table.to_csv(sig_table_path, index=False)

display(sig_table)

print("Saved thesis-ready significance table:")
print(sig_table_path)